In [ ]:
from pathlib import Path
import json

# Dataset: 250 synthetic FK-2c runs
DATA_ROOT = Path("../synthetic_runs1T_FK_2c")  # <-- CHANGE THIS if needed

samples = sorted([p for p in DATA_ROOT.glob("synthetic1T_run*") if p.is_dir()])
print("DATA_ROOT exists:", DATA_ROOT.exists())
print("Num samples:", len(samples))
print("First sample:", samples[0] if samples else None)

params = json.loads((samples[0] / "parameters.json").read_text())
print("Parameter keys:", list(params.keys()))
print("Example params:", params)

In [ ]:
# Parameters the model learns to predict
# Dw  = white-matter diffusion coefficient
# rho = tumour cell proliferation rate
TARGET_KEYS = ("Dw", "rho")

In [ ]:
import random
random.seed(42)
samples = list(samples)   # convert from sorted list to plain list for shuffle
random.shuffle(samples)

n = len(samples)
train = samples[:int(0.7*n)]
val   = samples[int(0.7*n):int(0.85*n)]
test  = samples[int(0.85*n):]

print("Split sizes:", len(train), len(val), len(test))

In [ ]:
import numpy as np
import json

def get_y(sample_dir):
    """Read raw targets for a sample, applying log to Dw for log-space training."""
    p = json.loads((sample_dir / "parameters.json").read_text())
    vals = []
    for k in TARGET_KEYS:
        v = float(p[k])
        if k == "Dw":
            v = np.log(v)   # train Dw in log-space; exponentiate back at inference
        vals.append(v)
    return np.array(vals, dtype=np.float32)

Y_train = np.stack([get_y(s) for s in train], axis=0)
Y_MEAN  = Y_train.mean(axis=0)
Y_STD   = Y_train.std(axis=0) + 1e-6

print("Y_MEAN:", Y_MEAN)
print("Y_STD :", Y_STD)

In [ ]:
import numpy as np
import SimpleITK as sitk
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
import json


def read_nii(path):
    img = sitk.ReadImage(str(path))
    arr = sitk.GetArrayFromImage(img).astype(np.float32)  # (D, H, W)
    return arr


def bbox_from_mask(mask, margin=12):
    """Return (z0,z1,y0,y1,x0,x1) tight bounding box with a fixed margin."""
    idx = np.argwhere(mask)
    if idx.size == 0:
        return None
    z0, y0, x0 = idx.min(axis=0)
    z1, y1, x1 = idx.max(axis=0) + 1
    z0 = max(z0 - margin, 0); y0 = max(y0 - margin, 0); x0 = max(x0 - margin, 0)
    z1 = min(z1 + margin, mask.shape[0])
    y1 = min(y1 + margin, mask.shape[1])
    x1 = min(x1 + margin, mask.shape[2])
    return (z0, z1, y0, y1, x0, x1)


def crop(arr, bb):
    z0, z1, y0, y1, x0, x1 = bb
    return arr[z0:z1, y0:y1, x0:x1]


def resize_3d(arr, out_shape=(96, 96, 96)):
    t = torch.from_numpy(arr)[None, None, ...]
    t = F.interpolate(t, size=out_shape, mode="trilinear", align_corners=False)
    return t[0, 0].numpy()


def normalize_channel(arr):
    m = arr.mean()
    s = arr.std()
    return (arr - m) / (s + 1e-6)


class TumorParamDatasetCached(Dataset):
    """
    Cached 4-channel physics dataset for the FK-2c synthetic runs.

    Input channels (physics mode):
        0: P.nii.gz   – proliferating tumour cells
        1: N.nii.gz   – necrotic core
        2: S.nii.gz   – surrounding tissue / edema response
        3: segm mask  – binary tumour boundary

    All volumes are cropped to the tumour bounding box, resized to
    `out_shape`, and z-score normalised (non-binary channels only).
    Targets (Dw, rho) are normalised using training-set statistics.
    Dw is trained in log-space to handle its log-normal distribution.
    """

    def __init__(self, sample_dirs, out_shape=(96, 96, 96),
                 target_keys=("Dw", "rho"), verbose=True):
        self.sample_dirs = sample_dirs
        self.out_shape   = out_shape
        self.target_keys = target_keys

        self.cache = []
        if verbose:
            print(f"Caching {len(sample_dirs)} samples... (one-time cost)")

        for i, sd in enumerate(self.sample_dirs):
            self.cache.append(self._load_one(sd))
            if verbose and (i + 1) % 25 == 0:
                print(f"  cached {i+1}/{len(sample_dirs)}")

        if verbose:
            print("Caching done.")

    def __len__(self):
        return len(self.cache)

    def __getitem__(self, idx):
        return self.cache[idx]

    def _load_one(self, sd):
        # ---- Targets ----
        params = json.loads((sd / "parameters.json").read_text())
        vals = []
        for k in self.target_keys:
            v = float(params[k])
            if k == "Dw":
                v = np.log(v)
            vals.append(v)
        y_raw  = np.array(vals, dtype=np.float32)
        y_norm = (y_raw - Y_MEAN) / Y_STD
        y      = torch.tensor(y_norm, dtype=torch.float32)

        # ---- Inputs (4-channel physics) ----
        segm = read_nii(sd / "segm.nii.gz")
        bb   = bbox_from_mask(segm > 0, margin=12)

        chans = [
            read_nii(sd / "P.nii.gz"),
            read_nii(sd / "N.nii.gz"),
            read_nii(sd / "S.nii.gz"),
            (segm > 0).astype(np.float32),
        ]

        processed = []
        for c in chans:
            if bb is not None:
                c = crop(c, bb)
            c = resize_3d(c, self.out_shape)
            if np.unique(c).size > 2:   # z-score non-binary channels
                c = normalize_channel(c)
            processed.append(c)

        x = torch.from_numpy(np.stack(processed, axis=0)).float()  # (4, D, H, W)
        return x, y

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler


# ---------------------------------------------------------------------------
# Building blocks
# ---------------------------------------------------------------------------

class ResBlock3D(nn.Module):
    """
    3-D residual block: two 3x3x3 convolutions with a projection shortcut.
    Using stride=2 on the first conv halves spatial resolution (replaces MaxPool).
    """

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv3d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm3d(out_ch)
        self.conv2 = nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm3d(out_ch)
        self.relu  = nn.ReLU(inplace=True)

        # Projection shortcut when channel count or spatial size changes
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv3d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm3d(out_ch),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + self.shortcut(x))


# ---------------------------------------------------------------------------
# Main model
# ---------------------------------------------------------------------------

class ResNet3DRegressor(nn.Module):
    """
    3-D ResNet-style regressor for tumour parameter estimation.

    Replaces the plain stacked-conv CNN3DRegressor with residual blocks,
    providing better gradient flow and feature reuse on the small (~250
    sample) synthetic dataset.  Stride-2 residual blocks are used instead
    of MaxPool so the shortcut path always retains spatial information.

    Architecture:
        Stem  : 3x3x3 Conv -> BN -> ReLU              (32 ch)
        Stage1: ResBlock3D(32  -> 64,  stride=2)       halves to 48^3
        Stage2: ResBlock3D(64  -> 128, stride=2)       halves to 24^3
        Stage3: ResBlock3D(128 -> 256, stride=2)       halves to 12^3
        Stage4: ResBlock3D(256 -> 512, stride=2)       halves to  6^3
        Head  : AdaptiveAvgPool -> FC(512->256) -> Dropout
                               -> FC(256->128) -> Dropout
                               -> FC(128->out_dim)
    """

    def __init__(self, in_channels=4, out_dim=2, dropout=0.3):
        super().__init__()

        # Stem: lightweight initial feature extraction
        self.stem = nn.Sequential(
            nn.Conv3d(in_channels, 32, 3, padding=1, bias=False),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
        )

        # Residual stages – each stride=2 block halves spatial resolution
        self.layer1 = ResBlock3D(32,  64,  stride=2)
        self.layer2 = ResBlock3D(64,  128, stride=2)
        self.layer3 = ResBlock3D(128, 256, stride=2)
        self.layer4 = ResBlock3D(256, 512, stride=2)

        # Regression head
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten(),
            nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 128), nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, out_dim),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return self.head(x)


# ---------------------------------------------------------------------------
# Setup
# ---------------------------------------------------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")

MODE      = "physics"
OUT_SHAPE = (96, 96, 96)

# Cache all splits once – amortises IO cost across epochs
train_ds = TumorParamDatasetCached(train, out_shape=OUT_SHAPE, target_keys=TARGET_KEYS, verbose=True)
val_ds   = TumorParamDatasetCached(val,   out_shape=OUT_SHAPE, target_keys=TARGET_KEYS, verbose=True)

NUM_WORKERS = 0
BATCH_SIZE  = 4   # reduce to 2 if GPU OOM

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

model   = ResNet3DRegressor(in_channels=4, out_dim=len(TARGET_KEYS)).to(device)
opt     = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scaler  = GradScaler("cuda")
loss_fn = nn.MSELoss()

# Cosine LR decay: warmup not needed for small runs; revisit for >300 epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=300, eta_min=1e-5)


def mae(pred, y):
    return (pred - y).abs().mean()


EPOCHS = 300

best_val_mse = float("inf")
best_state   = None
best_epoch   = -1

for epoch in range(1, EPOCHS + 1):
    # ---- Train ----
    model.train()
    tr_loss = tr_mae = 0.0
    for x, y in train_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        opt.zero_grad(set_to_none=True)
        with autocast("cuda"):
            pred = model(x)
            loss = loss_fn(pred, y)

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        tr_loss += loss.item()
        tr_mae  += mae(pred.detach(), y).item()

    scheduler.step()

    # ---- Validate ----
    model.eval()
    va_loss = va_mae = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            with autocast("cuda"):
                pred = model(x)
                loss = loss_fn(pred, y)
            va_loss += loss.item()
            va_mae  += mae(pred, y).item()

    val_mse = va_loss / len(val_loader)

    # Checkpoint best model
    if val_mse < best_val_mse:
        best_val_mse = val_mse
        best_epoch   = epoch
        best_state   = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    print(
        f"Epoch {epoch:03d} | "
        f"train MSE {tr_loss/len(train_loader):.4f} MAE {tr_mae/len(train_loader):.4f} | "
        f"val MSE {val_mse:.4f} MAE {va_mae/len(val_loader):.4f} | "
        f"best val {best_val_mse:.4f} @ {best_epoch:03d} | "
        f"lr {scheduler.get_last_lr()[0]:.2e}"
    )

# Restore best weights before evaluation
if best_state is not None:
    model.load_state_dict(best_state)
    print(f"\nLoaded best model from epoch {best_epoch} with val MSE {best_val_mse:.6f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import torch

test_ds     = TumorParamDatasetCached(test, out_shape=OUT_SHAPE, target_keys=TARGET_KEYS, verbose=True)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0, pin_memory=True)

model.eval()
preds_norm, trues_norm = [], []
with torch.no_grad():
    for x, y in test_loader:
        x    = x.to(device, non_blocking=True)
        pred = model(x).cpu().numpy()[0]
        preds_norm.append(pred)
        trues_norm.append(y.numpy()[0])

preds_norm = np.array(preds_norm)
trues_norm = np.array(trues_norm)

# Denormalise back to original parameter scale
preds = preds_norm * Y_STD + Y_MEAN
trues = trues_norm * Y_STD + Y_MEAN

# Undo log-transform for Dw
for i, k in enumerate(TARGET_KEYS):
    if k == "Dw":
        preds[:, i] = np.exp(preds[:, i])
        trues[:, i] = np.exp(trues[:, i])

for i, k in enumerate(TARGET_KEYS):
    mae_i  = np.abs(preds[:, i] - trues[:, i]).mean()
    rmse_i = np.sqrt(((preds[:, i] - trues[:, i])**2).mean())
    print(f"{k}: MAE={mae_i:.6f}  RMSE={rmse_i:.6f}")

for i, k in enumerate(TARGET_KEYS):
    plt.figure()
    plt.scatter(trues[:, i], preds[:, i])
    mn = min(trues[:, i].min(), preds[:, i].min())
    mx = max(trues[:, i].max(), preds[:, i].max())
    plt.plot([mn, mx], [mn, mx], "r--")
    plt.xlabel("True")
    plt.ylabel("Predicted")
    plt.title(f"{k}: Predicted vs True  (ResNet3DRegressor)")
    plt.tight_layout()
    plt.show()

In [ ]:
import torch

torch.save({
    "model_state": model.state_dict(),
    "target_keys": TARGET_KEYS,
    "mode":        MODE,
    "out_shape":   OUT_SHAPE,
    "y_mean":      Y_MEAN,
    "y_std":       Y_STD,
}, "dcnn_resnet_model.pt")

print("Saved -> dcnn_resnet_model.pt")